## Cell 1: Imports and Setup

The following cell will import necessary components from the existing project and external libraries. These should have been installed in the previous setup steps. If any modules are not found, ensure that `npm install` has been run in the project root and that the ijavascript kernel is correctly installed and selected.

In [ ]:
import { graph } from '../src/agents/research/graph';
import { State as StateAnnotation } from '../src/agents/research/state'; // Assuming State is the correct export
// import { system_prompt } from '../src/agents/research/prompts'; // system_prompt might not be directly used by the notebook, but by the agent
// import { getChatModel } from '../src/llm'; // getChatModel is used internally by the agent
import { HumanMessage } from '@langchain/core/messages';

// For logging within the notebook environment if needed
// import { Logger } from 'tslog'; 
// const log: Logger = new Logger({ name: 'MigrationAnalysisNotebook' });

console.log("Notebook setup complete and modules imported.");

// Define a type alias for the State if StateAnnotation is a class/const with a State member
// For now, we'll assume StateAnnotation can be used to type the state object.
type State = StateAnnotation;

## Cell 2: Define the Migration Scenario

In [ ]:
const migrationScenario = "Log4J migration from version 1 to 2";
console.log(`Migration Scenario: ${migrationScenario}`);

## Cell 3: Define the Extraction Schema

This schema guides the research agent on what information to extract. The properties are based on the hints in `data/prompts/migration_research_agent_v1.md`.

In [ ]:
interface MigrationConcern {
    name: string;
    description: string;
    detectionMethod: string;
    mitigationStrategy: string;                
    fixHint?: string;
    exampleUnresolved: string;
    exampleResolved: string;
}

const extractionSchema = {
    type: "object",
    properties: {
        concerns: {
            type: "array",
            items: {
                type: "object",
                properties: {
                    name: { type: "string", description: "Unique name for the concern" },
                    description: { type: "string", description: "Detailed description of the concern" },
                    detectionMethod: { type: "string", description: "How to detect if this concern exists" },
                    mitigationStrategy: { type: "string", description: "Recommended mitigation strategy" },
                    fixHint: { type: "string", description: "Hint on how to fix this (optional)" },
                    exampleUnresolved: { type: "string", description: "Code example of the unresolved situation" },
                    exampleResolved: { type: "string", description: "Code example of the resolved situation" }
                },
                required: ["name", "description", "detectionMethod", "mitigationStrategy", "exampleUnresolved", "exampleResolved"]
            }
        }
    },
    required: ["concerns"]
};
console.log("Extraction Schema defined."); // No need to print the whole schema here, it's verbose
// console.log("Extraction Schema defined:", JSON.stringify(extractionSchema, null, 2));

## Cell 4: Load LLM Configuration

The agent requires LLM configuration to interact with a language model. This configuration is loaded from `../data/config.yaml`. 
Please ensure this file exists and is correctly set up. 
You can typically find a sample or create one based on the following structure:

```yaml
# Example ../data/config.yaml
llm:
  activeProvider: bedrock # or openai, anthropic, google, xai, groq
  providers:
    bedrock:
      type: bedrock
      region: "us-east-1" # Change to your AWS region
      model: "anthropic.claude-3-sonnet-20240229-v1:0" # Example model
      # credentials can be configured if needed, otherwise AWS SDK defaults are used
    openai:
      type: openai
      model: "gpt-4-turbo-preview"
      apiKey: "YOUR_OPENAI_API_KEY" # Replace with your actual key or set OPENAI_API_KEY environment variable
    # ... other provider configurations
```

The cell below will attempt to load this configuration.

In [ ]:
import { ConfigLoader, type RulegenixConfig } from '../src/config/loader'; 
import { getActiveProvider, type ProviderConfig } from '../src/config/config'; 

let llmConfig: RulegenixConfig;
let activeProviderConfig: ProviderConfig;

try {
    // Path is relative to the project root, assuming Jupyter is started there.
    const configLoader = ConfigLoader.getInstance("../data/config.yaml"); 
    llmConfig = configLoader.getConfig();
    activeProviderConfig = getActiveProvider(llmConfig);
    console.log("LLM Configuration loaded successfully.");
    console.log("Active provider:", llmConfig.llm.activeProvider);
    console.log("Model:", activeProviderConfig.model);
} catch (error) {
    console.error("Failed to load LLM configuration from ../data/config.yaml:", error.message);
    console.log("Please ensure ../data/config.yaml exists and is correctly formatted.");
    llmConfig = { 
        llm: { 
            activeProvider: "fallback", 
            providers: { 
                fallback: { type: "openai", model: "gpt-3.5-turbo", apiKey:"YOUR_KEY_HERE" } 
            } 
        }
    } as RulegenixConfig; // Cast to RulegenixConfig to satisfy type checking
    activeProviderConfig = getActiveProvider(llmConfig);
    console.warn("Using a fallback LLM configuration. Please fix config.yaml for the agent to work.");
}

## Cell 5: Invoke the Migration Research Agent

This cell contains the logic to prepare the initial state and run the agent using the loaded configuration.

In [ ]:
// graph, StateAnnotation, HumanMessage should be imported in Cell 1
// migrationScenario, extractionSchema, llmConfig, activeProviderConfig are defined in previous cells

async function runAgent() {
    console.log("Preparing to run the agent with loaded configuration...");

    if (!llmConfig || llmConfig.llm.activeProvider === "fallback") {
        console.error("LLM Configuration is not properly loaded or is a fallback. Agent cannot run effectively.");
        console.error("Please ensure '../data/config.yaml' is correctly set up and reload the configuration cell (Cell 4).");
        return { message: "Agent run aborted due to missing/fallback LLM configuration." };
    }
    
    const initialState: State = {
        messages: [new HumanMessage(`Research migration issues for: ${migrationScenario}`)],
        migrationScenario: migrationScenario,
        extractionSchema: extractionSchema, // Pass the object directly, agent stringifies it internally
        llmConfig: llmConfig, 
        maxSearchResults: 3, 
        loopStep: 0,
        info: {} // Initialize info object, critical for the agent's operation
        // searchQueries, searchResults are not initialized here, they are populated by the agent.
    };

    console.log("Initial state for agent:", initialState);

    let finalExtractedInfo: any = null; // Use 'any' for flexibility, or a more specific type if known
    let iterationCount = 0;
    const maxIterations = 15; // Increased safeguard for streaming, can be adjusted

    try {
        const stream = await graph.stream(initialState);
        for await (const event of stream) {
            iterationCount++;
            // console.log(`Agent Event [${iterationCount}]:`, JSON.stringify(event, null, 2)); // Can be very verbose
            
            // Log specific parts of the event for better readability
            if (event.plan) console.log(`Event [${iterationCount}] - Plan:`, event.plan.plan);
            if (event.search) console.log(`Event [${iterationCount}] - Search: Queries -`, event.search.searchQueries, `Results -`, event.search.searchResults ? event.search.searchResults.length : 0);
            if (event.callModel) {
                console.log(`Event [${iterationCount}] - CallModel: Last message -`, event.callModel.messages[event.callModel.messages.length -1].pretty());
                if (event.callModel.info && Object.keys(event.callModel.info).length > 0) {
                    console.log("Extracted info found in 'callModel' event's 'info' field:", event.callModel.info);
                    finalExtractedInfo = event.callModel.info;
                }
                // Check for tool calls specifically named 'Info'
                const lastMessage = event.callModel.messages[event.callModel.messages.length - 1];
                if (lastMessage && lastMessage.tool_calls && lastMessage.tool_calls.length > 0) {
                    const infoToolCall = lastMessage.tool_calls.find(tc => tc.name === 'Info');
                    if (infoToolCall) {
                        finalExtractedInfo = infoToolCall.args;
                        console.log("Extracted info found in 'callModel' event via 'Info' tool_call:", finalExtractedInfo);
                    }
                }
            }
            if (event.__end__) {
                console.log(`Event [${iterationCount}] - End: Final state -`, event.__end__);
                if (event.__end__.info && Object.keys(event.__end__.info).length > 0) {
                     console.log("Extracted info found in '__end__' event's 'info' field:", event.__end__.info);
                    finalExtractedInfo = event.__end__.info;
                } else if (event.__end__.messages) {
                    // Check the last message in the __end__ state for tool calls
                    const lastMessage = event.__end__.messages[event.__end__.messages.length - 1];
                    if (lastMessage && lastMessage.tool_calls && lastMessage.tool_calls.length > 0) {
                        const infoToolCall = lastMessage.tool_calls.find(tc => tc.name === 'Info');
                        if (infoToolCall) {
                            finalExtractedInfo = infoToolCall.args;
                            console.log("Extracted info found in '__end__' event via 'Info' tool_call:", finalExtractedInfo);
                        }
                    }
                }
            }

            if (iterationCount >= maxIterations) {
                console.warn("Reached max iterations, stopping stream processing.");
                break;
            }
        }

        console.log("\n--- Agent Stream Completed ---");
        if (finalExtractedInfo && Object.keys(finalExtractedInfo).length > 0) {
            console.log("Final Extracted Information:");
            console.log(JSON.stringify(finalExtractedInfo, null, 2));
            // To display pretty JSON in ijavascript output:
            // import { display, HTML } from 'ijavascript'; // Make sure ijavascript is available
            // display(HTML("<pre style='white-space: pre-wrap; word-wrap: break-word;'>" + JSON.stringify(finalExtractedInfo, null, 2) + "</pre>"));
            return finalExtractedInfo;
        } else {
            console.log("No specific information extracted by the agent or not found in the expected event structure.");
            return { message: "No information extracted. Check agent logs and events above." };
        }
    } catch (error) {
        console.error("Error running agent stream:", error);
        return { error: "Agent execution failed.", details: error.message, stack: error.stack };
    }
}

// Automatically run the agent when this cell is executed.
// The output (finalExtractedInfo or error) will be logged above.
runAgent().then(output => {
    console.log("Agent run process finished. See logs above for details and extracted information.");
    // If you want to see the final output again here, you can log it:
    // console.log("Final output from runAgent:", JSON.stringify(output, null, 2));
}).catch(error => {
    console.error("Unhandled error during agent execution from notebook:", error);
});

## Cell 6: Display Results

Agent output, including intermediate steps and any errors, will be displayed in the console log of the cell above (Cell 5) after running the invocation. The final extracted information (JSON) will also be printed there if the agent completes successfully and extracts data.